
# An Explainable Physics-Informed Digital Twin
## for Lithium-Ion Battery Health Monitoring

### Project Objective

This project aims to develop a lightweight,
physics-informed Digital Twin prototype for
monitoring lithium-ion battery degradation.

The framework combines:
- Experimental battery cycling data
- Battery health indicator extraction
- Equivalent-circuit modelling
- Machine learning-based SoH estimation
- Digital Twin state monitoring

### Dataset

NASA Lithium-Ion Battery Aging Dataset

Initial case study: Battery B0005

### Research Question

To what extent can physically interpretable
health indicators and equivalent-circuit modelling,
combined with lightweight machine learning,
improve the accuracy and interpretability of
State-of-Health monitoring in lithium-ion batteries?


In [ ]:

# Check the Python environment

import sys

print("Python version:", sys.version)
print("Development environment: Google Colab")
print("Battery Digital Twin project initialized.")


In [ ]:

# Import the libraries required for battery data analysis

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat

print("All libraries imported successfully!")


In [ ]:

from google.colab import files

uploaded = files.upload()


In [ ]:

# Load the NASA battery dataset

mat = loadmat(
    "B0005.mat",
    squeeze_me=True,
    struct_as_record=False
)

print("Dataset loaded successfully!")


In [ ]:

# Identify the main variables stored in the MATLAB file

for key in mat.keys():
    if not key.startswith("__"):
        print(key)


In [ ]:

# Access the main battery structure

battery = mat["B0005"]

# Display the available fields
print("Battery structure fields:")
print(battery._fieldnames)


In [ ]:

# Access the experimental cycle records

cycles = np.atleast_1d(battery.cycle)

print("Total number of experimental records:", len(cycles))


In [ ]:

# Inspect the first experimental record

first_record = cycles[0]

print("Fields in the first record:")
print(first_record._fieldnames)


In [ ]:

# Inspect the metadata of the first experiment

print("Experiment type:", first_record.type)

print("Ambient temperature:",
      first_record.ambient_temperature)

print("Experiment time:",
      first_record.time)


In [ ]:

# Inspect the internal structure of the experimental data

experiment_data = first_record.data

print("Fields inside experimental data:")
print(experiment_data._fieldnames)


In [ ]:

# Inspect the size and shape of each measurement

for field in experiment_data._fieldnames:
    values = getattr(experiment_data, field)

    print(
        field,
        "| Shape:", np.shape(values),
        "| Number of measurements:", np.size(values)
    )


In [ ]:
time = experiment_data.Time
voltage = experiment_data.Voltage_measured
current = experiment_data.Current_measured
temperature = experiment_data.Temperature_measured

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axes[0].plot(time, voltage)
axes[0].set_ylabel("Voltage (V)")
axes[0].set_title("First Charge Record — B0005")

axes[1].plot(time, current)
axes[1].set_ylabel("Current (A)")

axes[2].plot(time, temperature)
axes[2].set_ylabel("Temperature (°C)")
axes[2].set_xlabel("Time (seconds)")

plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter

record_types = [
    str(record.type) for record in cycles
]

type_counts = Counter(record_types)

for record_type, count in type_counts.items():
    print(f"{record_type}: {count}")

In [ ]:
discharge_records = [
    record for record in cycles
    if str(record.type) == "discharge"
]

first_discharge = discharge_records[0]

print("Number of discharge records:", len(discharge_records))
print("Discharge data fields:")
print(first_discharge.data._fieldnames)

In [ ]:
discharge_data = first_discharge.data

for field in discharge_data._fieldnames:
    values = getattr(discharge_data, field)

    print(
        field,
        "| Shape:", np.shape(values),
        "| First 5 values:", np.asarray(values).flatten()[:5]
    )

In [ ]:
capacity_values = np.array([
    float(record.data.Capacity)
    for record in discharge_records
])

print("Number of capacity measurements:", len(capacity_values))
print("First 10 capacity values:")
print(capacity_values[:10])

print("\nInitial capacity:", capacity_values[0], "Ah")
print("Minimum capacity:", capacity_values.min(), "Ah")
print("Maximum capacity:", capacity_values.max(), "Ah")

In [ ]:
initial_capacity = capacity_values[0]

soh_values = (capacity_values / initial_capacity) * 100

print("First 10 SoH values:")
print(np.round(soh_values[:10], 2))

print("\nInitial SoH:", round(soh_values[0], 2), "%")
print("Minimum SoH:", round(soh_values.min(), 2), "%")
print("Final SoH:", round(soh_values[-1], 2), "%")

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    range(1, len(soh_values) + 1),
    soh_values,
    marker="o",
    markersize=3,
    linewidth=1.5
)

plt.axhline(
    y=70,
    linestyle="--",
    label="70% SoH reference"
)

plt.xlabel("Discharge Record Index")
plt.ylabel("State of Health (%)")
plt.title("Battery B0005 — State of Health Degradation")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
print("First discharge record:")
print("Date/time:", discharge_records[0].time)

print("\nLast discharge record:")
print("Date/time:", discharge_records[-1].time)

In [ ]:
from datetime import datetime

def convert_timestamp(timestamp):
    values = np.asarray(timestamp).flatten()
    return datetime(
        int(values[0]),
        int(values[1]),
        int(values[2]),
        int(values[3]),
        int(values[4]),
        int(values[5])
    )

timestamps = [
    convert_timestamp(record.time)
    for record in discharge_records
]

is_chronological = all(
    timestamps[i] <= timestamps[i + 1]
    for i in range(len(timestamps) - 1)
)

print("Records are chronologically ordered:", is_chronological)
print("First timestamp:", timestamps[0])
print("Last timestamp:", timestamps[-1])

In [ ]:
battery_df = pd.DataFrame({
    "record_index": range(1, len(discharge_records) + 1),
    "timestamp": timestamps,
    "capacity_Ah": capacity_values,
    "SoH_percent": soh_values
})

print(battery_df.head(10))
print("\nDataset information:")
print(battery_df.info())

In [ ]:
print("Missing values:")
print(battery_df.isnull().sum())

print("\nDuplicated rows:")
print(battery_df.duplicated().sum())

print("\nStatistical summary:")
print(battery_df[["capacity_Ah", "SoH_percent"]].describe())

In [ ]:
features = []

for i, record in enumerate(discharge_records):

    data = record.data

    voltage = np.asarray(data.Voltage_measured)
    current = np.asarray(data.Current_measured)
    temperature = np.asarray(data.Temperature_measured)
    time = np.asarray(data.Time)

    features.append({
        "record_index": i + 1,
        "voltage_mean": np.mean(voltage),
        "voltage_min": np.min(voltage),
        "voltage_max": np.max(voltage),
        "current_mean": np.mean(current),
        "temperature_mean": np.mean(temperature),
        "temperature_max": np.max(temperature),
        "discharge_duration_s": time[-1] - time[0]
    })

features_df = pd.DataFrame(features)

print(features_df.head())

In [ ]:
battery_analysis_df = battery_df.merge(
    features_df,
    on="record_index"
)

print("Dataset shape:", battery_analysis_df.shape)
print("\nColumns:")
print(battery_analysis_df.columns.tolist())

print("\nFirst five records:")
display(battery_analysis_df.head())

In [ ]:
analysis_features = [
    "voltage_mean",
    "voltage_min",
    "voltage_max",
    "current_mean",
    "temperature_mean",
    "temperature_max",
    "discharge_duration_s"
]

pearson_corr = battery_analysis_df[
    analysis_features + ["SoH_percent"]
].corr(method="pearson")["SoH_percent"].drop("SoH_percent")

spearman_corr = battery_analysis_df[
    analysis_features + ["SoH_percent"]
].corr(method="spearman")["SoH_percent"].drop("SoH_percent")

correlation_df = pd.DataFrame({
    "Pearson": pearson_corr,
    "Spearman": spearman_corr
})

print(correlation_df.sort_values(
    by="Spearman",
    key=abs,
    ascending=False
))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

plots = [
    ("voltage_mean", "Mean Voltage (V)"),
    ("temperature_max", "Maximum Temperature (°C)"),
    ("temperature_mean", "Mean Temperature (°C)")
]

for ax, (column, label) in zip(axes, plots):
    ax.scatter(
        battery_analysis_df[column],
        battery_analysis_df["SoH_percent"],
        alpha=0.65
    )

    ax.set_xlabel(label)
    ax.set_ylabel("SoH (%)")
    ax.set_title(f"SoH vs {label}")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

variables = [
    ("voltage_mean", "Mean Voltage (V)"),
    ("temperature_max", "Maximum Temperature (°C)"),
    ("temperature_mean", "Mean Temperature (°C)")
]

for ax, (column, label) in zip(axes, variables):

    ax.plot(
        battery_analysis_df["record_index"],
        battery_analysis_df[column],
        marker=".",
        markersize=4
    )

    ax.set_ylabel(label)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Discharge Record Index")

fig.suptitle("Evolution of Battery Indicators During Ageing")
plt.tight_layout()
plt.show()

In [ ]:
for i in range(45, 52):

    record = discharge_records[i]
    data = record.data

    temperature = np.asarray(data.Temperature_measured)

    print(f"\nRecord {i+1}")
    print("Timestamp:", convert_timestamp(record.time))
    print("Ambient temperature:", record.ambient_temperature)
    print("Capacity:", record.data.Capacity)
    print("Temperature mean:", np.mean(temperature))
    print("Temperature min:", np.min(temperature))
    print("Temperature max:", np.max(temperature))

In [ ]:
plt.figure(figsize=(10, 5))

for i in [46, 47, 48, 49]:
    data = discharge_records[i].data

    plt.plot(
        data.Time,
        data.Temperature_measured,
        label=f"Record {i+1}"
    )

plt.xlabel("Time (seconds)")
plt.ylabel("Temperature (°C)")
plt.title("Temperature Profiles Around the Observed Transition")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
battery_analysis_df["temperature_rise"] = (
    battery_analysis_df["temperature_max"]
    - battery_analysis_df["temperature_mean"]
)

In [ ]:
battery_analysis_df["temperature_rise"] = [
    np.max(record.data.Temperature_measured)
    - np.min(record.data.Temperature_measured)
    for record in discharge_records
]

print(
    battery_analysis_df[
        ["record_index", "SoH_percent", "temperature_mean",
         "temperature_max", "temperature_rise"]
    ].head(10)
)

In [ ]:
temperature_correlations = battery_analysis_df[
    ["temperature_mean", "temperature_max",
     "temperature_rise", "SoH_percent"]
].corr(method="pearson")["SoH_percent"]

temperature_spearman = battery_analysis_df[
    ["temperature_mean", "temperature_max",
     "temperature_rise", "SoH_percent"]
].corr(method="spearman")["SoH_percent"]

print(pd.DataFrame({
    "Pearson": temperature_correlations.drop("SoH_percent"),
    "Spearman": temperature_spearman.drop("SoH_percent")
}))

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    battery_analysis_df["temperature_rise"],
    battery_analysis_df["SoH_percent"],
    alpha=0.7
)

plt.xlabel("Temperature Rise (°C)")
plt.ylabel("State of Health (%)")
plt.title("Battery SoH vs Temperature Rise")
plt.grid(alpha=0.3)

plt.show()

In [ ]:
temperature_outliers = battery_analysis_df[
    (battery_analysis_df["temperature_rise"] > 15.8) &
    (battery_analysis_df["SoH_percent"] < 75)
]

print(
    temperature_outliers[
        ["record_index", "SoH_percent",
         "temperature_rise", "temperature_mean",
         "temperature_max"]
    ].to_string(index=False)
)

In [ ]:
records_to_compare = [165, 166, 167]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, i in zip(axes, records_to_compare):

    data = discharge_records[i].data

    ax.plot(data.Time, data.Temperature_measured)

    ax.set_title(f"Record {i+1}")
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Temperature (°C)")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
battery_analysis_df.to_csv(
    "B0005_initial_feature_dataset.csv",
    index=False
)

print("Dataset saved successfully.")
print("Shape:", battery_analysis_df.shape)

In [ ]:
impedance_records = [
    record for record in cycles
    if str(record.type) == "impedance"
]

print("Number of impedance records:", len(impedance_records))

first_impedance = impedance_records[0]

print("\nImpedance record fields:")
print(first_impedance._fieldnames)

print("\nImpedance data fields:")
print(first_impedance.data._fieldnames)

In [ ]:
impedance_data = first_impedance.data

for field in impedance_data._fieldnames:
    values = getattr(impedance_data, field)

    print(
        field,
        "| Shape:", np.shape(values),
        "| Data type:", np.asarray(values).dtype,
        "| First 5 values:", np.asarray(values).flatten()[:5]
    )

In [ ]:
impedance_data_list = []

for i, record in enumerate(impedance_records):

    data = record.data

    impedance_data_list.append({
        "impedance_index": i + 1,
        "timestamp": convert_timestamp(record.time),
        "Re": float(data.Re),
        "Rct": float(data.Rct)
    })

impedance_df = pd.DataFrame(impedance_data_list)

print("Dataset shape:", impedance_df.shape)
print("\nFirst 10 records:")
display(impedance_df.head(10))

print("\nStatistical summary:")
display(impedance_df[["Re", "Rct"]].describe())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

axes[0].plot(
    impedance_df["timestamp"],
    impedance_df["Re"],
    marker=".",
    markersize=3
)
axes[0].set_ylabel("Re")
axes[0].set_title("Evolution of Ohmic Resistance")
axes[0].grid(alpha=0.3)

axes[1].plot(
    impedance_df["timestamp"],
    impedance_df["Rct"],
    marker=".",
    markersize=3
)
axes[1].set_ylabel("Rct")
axes[1].set_title("Evolution of Charge-Transfer Resistance")
axes[1].set_xlabel("Experimental Date")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
impedance_df["time_difference_hours"] = (
    impedance_df["timestamp"].diff().dt.total_seconds() / 3600
)

print("Time difference statistics (hours):")
print(impedance_df["time_difference_hours"].describe())

print("\nLargest time gaps:")
display(
    impedance_df.nlargest(
        10, "time_difference_hours"
    )[["timestamp", "time_difference_hours", "Re", "Rct"]]
)

In [ ]:
long_gaps = impedance_df[
    impedance_df["time_difference_hours"] > 6
]

print("Number of gaps greater than 6 hours:",
      len(long_gaps))

print("\nLong gaps:")
display(
    long_gaps[
        ["timestamp", "time_difference_hours", "Re", "Rct"]
    ]
)

In [ ]:
discharge_time_df = battery_analysis_df[
    ["timestamp", "record_index", "SoH_percent"]
].rename(columns={"timestamp": "discharge_timestamp"})

impedance_time_df = impedance_df.copy()

matched_df = pd.merge_asof(
    impedance_time_df.sort_values("timestamp"),
    discharge_time_df.sort_values("discharge_timestamp"),
    left_on="timestamp",
    right_on="discharge_timestamp",
    direction="nearest"
)

matched_df["time_difference_hours"] = (
    matched_df["timestamp"] -
    matched_df["discharge_timestamp"]
).abs().dt.total_seconds() / 3600

print("Matching time-difference statistics:")
print(matched_df["time_difference_hours"].describe())

print("\nFirst 10 matches:")
display(matched_df.head(10))

In [ ]:
print("Total impedance records:", len(matched_df))

print(
    "Unique discharge records matched:",
    matched_df["record_index"].nunique()
)

print(
    "Impedance records per discharge record:"
)

print(
    matched_df["record_index"]
    .value_counts()
    .value_counts()
    .sort_index()
)

In [ ]:
impedance_aggregated = (
    matched_df
    .groupby("record_index")
    .agg(
        Re_mean=("Re", "mean"),
        Rct_mean=("Rct", "mean"),
        impedance_count=("impedance_index", "count")
    )
    .reset_index()
)

print("Aggregated dataset shape:",
      impedance_aggregated.shape)

display(impedance_aggregated.head(10))

In [ ]:
battery_impedance_df = battery_analysis_df.merge(
    impedance_aggregated,
    on="record_index",
    how="left"
)

print("Final dataset shape:", battery_impedance_df.shape)

print(
    "Records with impedance data:",
    battery_impedance_df["Re_mean"].notna().sum()
)

In [ ]:
matched_df["signed_offset_hours"] = (
    matched_df["timestamp"] -
    matched_df["discharge_timestamp"]
).dt.total_seconds() / 3600

print("Signed temporal offset (hours):")
print(matched_df["signed_offset_hours"].describe())

print("\nTemporal relationship:")
print(
    "Impedance before discharge:",
    (matched_df["signed_offset_hours"] < 0).sum()
)

print(
    "Impedance after discharge:",
    (matched_df["signed_offset_hours"] > 0).sum()
)

print(
    "Same timestamp:",
    (matched_df["signed_offset_hours"] == 0).sum()
)

In [ ]:
past_only_df = pd.merge_asof(
    discharge_time_df.sort_values("discharge_timestamp"),
    impedance_time_df.sort_values("timestamp"),
    left_on="discharge_timestamp",
    right_on="timestamp",
    direction="backward"
)

past_only_df["impedance_age_hours"] = (
    past_only_df["discharge_timestamp"]
    - past_only_df["timestamp"]
).dt.total_seconds() / 3600

print("Discharge records:", len(past_only_df))

print(
    "Records with a previous impedance measurement:",
    past_only_df["Re"].notna().sum()
)

print("\nAge of previous impedance measurement (hours):")
print(past_only_df["impedance_age_hours"].describe())

In [ ]:
tolerances = [1, 2, 6]

print("Matching tolerance analysis")
print("----------------------------")

for hours in tolerances:
    valid_matches = (
        past_only_df["impedance_age_hours"] <= hours
    ).sum()

    print(
        f"Tolerance: {hours} hours | "
        f"Valid matches: {valid_matches} / 168"
    )

In [ ]:
# Keep only impedance measurements within 1 hour
battery_impedance_final = past_only_df[
    past_only_df["impedance_age_hours"] <= 1
].copy()

# Keep the relevant impedance features
battery_impedance_final = battery_impedance_final[
    [
        "record_index",
        "Re",
        "Rct",
        "impedance_age_hours"
    ]
]

# Merge with the battery health indicators
final_df = battery_analysis_df.merge(
    battery_impedance_final,
    on="record_index",
    how="left"
)

print("Final dataset shape:", final_df.shape)
print("Matched impedance records:", final_df["Re"].notna().sum())
print("Missing impedance records:", final_df["Re"].isna().sum())

final_df.head()

In [ ]:
# Identify cycles with and without impedance measurements
final_df["impedance_available"] = final_df["Re"].notna()

# Summarise availability
availability_summary = (
    final_df.groupby("impedance_available")
    .agg(
        number_of_cycles=("record_index", "count"),
        mean_SoH=("SoH_percent", "mean"),
        min_SoH=("SoH_percent", "min"),
        max_SoH=("SoH_percent", "max")
    )
)

availability_summary

In [ ]:
plt.figure(figsize=(12, 4))

plt.scatter(
    final_df["record_index"],
    final_df["SoH_percent"],
    c=final_df["impedance_available"].map({
        True: "green",
        False: "red"
    }),
    s=35
)

plt.xlabel("Discharge cycle")
plt.ylabel("State of Health (%)")
plt.title("Impedance Availability Across Battery Ageing")
plt.grid(alpha=0.3)

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(
    battery_impedance_df["SoH_percent"],
    battery_impedance_df["Re_mean"]
)
axes[0].set_xlabel("SoH (%)")
axes[0].set_ylabel("Re")
axes[0].set_title("Ohmic Resistance vs Battery Health")
axes[0].grid(alpha=0.3)

axes[1].scatter(
    battery_impedance_df["SoH_percent"],
    battery_impedance_df["Rct_mean"]
)
axes[1].set_xlabel("SoH (%)")
axes[1].set_ylabel("Rct")
axes[1].set_title("Charge-transfer Resistance vs Battery Health")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import pearsonr, spearmanr

impedance_valid = battery_impedance_df.dropna(
    subset=["Re_mean", "Rct_mean"]
)

for feature in ["Re_mean", "Rct_mean"]:
    pearson_corr, pearson_p = pearsonr(
        impedance_valid[feature],
        impedance_valid["SoH_percent"]
    )

    spearman_corr, spearman_p = spearmanr(
        impedance_valid[feature],
        impedance_valid["SoH_percent"]
    )

    print(f"\n{feature}")
    print(f"Pearson correlation:  {pearson_corr:.4f} (p={pearson_p:.4g})")
    print(f"Spearman correlation: {spearman_corr:.4f} (p={spearman_p:.4g})")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7))

axes[0].plot(
    impedance_valid["record_index"],
    impedance_valid["Re_mean"],
    marker="o",
    markersize=3
)
axes[0].set_ylabel("Re")
axes[0].set_title("Ohmic Resistance Evolution")
axes[0].grid(alpha=0.3)

axes[1].plot(
    impedance_valid["record_index"],
    impedance_valid["Rct_mean"],
    marker="o",
    markersize=3
)
axes[1].set_xlabel("Discharge Cycle")
axes[1].set_ylabel("Rct")
axes[1].set_title("Charge-transfer Resistance Evolution")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Extract ambient temperature for each impedance record
impedance_ambient = []

for i, record in enumerate(impedance_records):
    impedance_ambient.append({
        "impedance_index": i + 1,
        "ambient_temperature": float(record.ambient_temperature)
    })

impedance_temp_df = pd.DataFrame(impedance_ambient)

# Merge with impedance measurements
impedance_temp_df = impedance_df.merge(
    impedance_temp_df,
    on="impedance_index"
)

# Plot resistance against ambient temperature
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(
    impedance_temp_df["ambient_temperature"],
    impedance_temp_df["Re"]
)
axes[0].set_xlabel("Ambient Temperature (°C)")
axes[0].set_ylabel("Re")
axes[0].set_title("Re vs Ambient Temperature")
axes[0].grid(alpha=0.3)

axes[1].scatter(
    impedance_temp_df["ambient_temperature"],
    impedance_temp_df["Rct"]
)
axes[1].set_xlabel("Ambient Temperature (°C)")
axes[1].set_ylabel("Rct")
axes[1].set_title("Rct vs Ambient Temperature")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(impedance_temp_df["ambient_temperature"].describe())

In [ ]:
# Inspect the first impedance measurement
first_impedance = impedance_records[0].data

print("Available impedance fields:")
print(first_impedance._fieldnames)

print("\nSense current:")
print(first_impedance.Sense_current[:5])

print("\nBattery impedance:")
print(first_impedance.Battery_impedance[:5])

print("\nRe:", first_impedance.Re)
print("Rct:", first_impedance.Rct)

In [ ]:
# Inspect dimensions of complex impedance measurements
for field in [
    "Sense_current",
    "Battery_current",
    "Current_ratio",
    "Battery_impedance",
    "Rectified_Impedance"
]:
    values = np.asarray(getattr(first_impedance, field))

    print(
        f"{field}: shape={values.shape}, "
        f"dtype={values.dtype}"
    )

In [ ]:
# Inspect the first impedance measurement's complex values
Z = np.asarray(first_impedance.Battery_impedance)

print("Real component:", np.real(Z)[:10])
print("Imaginary component:", np.imag(Z)[:10])
print("Magnitude:", np.abs(Z)[:10])

In [ ]:
Z = np.asarray(first_impedance.Battery_impedance)

plt.figure(figsize=(7, 6))

plt.plot(
    np.real(Z),
    -np.imag(Z),
    marker="o",
    markersize=5
)

plt.xlabel("Real Impedance, Z' ")
plt.ylabel("-Imaginary Impedance, -Z''")
plt.title("Battery Impedance Response — First Measurement")
plt.grid(alpha=0.3)
plt.axis("equal")

plt.show()

In [ ]:
# Inspect the sequence of impedance measurements
impedance_components = pd.DataFrame({
    "measurement_index": range(1, len(Z) + 1),
    "Z_real": np.real(Z),
    "Z_imaginary": np.imag(Z),
    "Z_magnitude": np.abs(Z)
})

impedance_components.head(15)

In [ ]:
raw_Z = np.asarray(first_impedance.Battery_impedance)
rectified_Z = np.asarray(first_impedance.Rectified_Impedance)

print("Raw impedance:")
print("Length:", len(raw_Z))
print("First 10:", raw_Z[:10])

print("\nRectified impedance:")
print("Length:", len(rectified_Z))
print("First 10:", rectified_Z[:10])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(np.real(raw_Z), -np.imag(raw_Z), marker="o")
axes[0].set_title("Raw Impedance")
axes[0].set_xlabel("Real impedance (Ω)")
axes[0].set_ylabel("-Imaginary impedance (Ω)")
axes[0].grid(alpha=0.3)
axes[0].axis("equal")

axes[1].plot(
    np.real(rectified_Z),
    -np.imag(rectified_Z),
    marker="o"
)
axes[1].set_title("Rectified Impedance")
axes[1].set_xlabel("Real impedance (Ω)")
axes[1].set_ylabel("-Imaginary impedance (Ω)")
axes[1].grid(alpha=0.3)
axes[1].axis("equal")

plt.tight_layout()
plt.show()

In [ ]:
rectified_features = []

for i, record in enumerate(impedance_records):
    Z_rect = np.asarray(record.data.Rectified_Impedance)

    rectified_features.append({
        "impedance_index": i + 1,
        "rectified_real_mean": np.mean(np.real(Z_rect)),
        "rectified_real_std": np.std(np.real(Z_rect)),
        "rectified_imag_mean": np.mean(np.imag(Z_rect)),
        "rectified_magnitude_mean": np.mean(np.abs(Z_rect))
    })

rectified_df = pd.DataFrame(rectified_features)

rectified_df.describe()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7))

axes[0].plot(
    rectified_df["impedance_index"],
    rectified_df["rectified_real_mean"]
)
axes[0].set_title("Mean Real Rectified Impedance")
axes[0].set_ylabel("Impedance (Ω)")
axes[0].grid(alpha=0.3)

axes[1].plot(
    rectified_df["impedance_index"],
    rectified_df["rectified_imag_mean"]
)
axes[1].set_title("Mean Imaginary Rectified Impedance")
axes[1].set_xlabel("Impedance Measurement Index")
axes[1].set_ylabel("Impedance (Ω)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Merge rectified impedance features with the existing impedance dataset
impedance_comparison = impedance_df.merge(
    rectified_df[
        [
            "impedance_index",
            "rectified_real_mean",
            "rectified_real_std",
            "rectified_imag_mean"
        ]
    ],
    on="impedance_index"
)

# Calculate correlations
correlation_columns = [
    "Re",
    "Rct",
    "rectified_real_mean",
    "rectified_real_std",
    "rectified_imag_mean"
]

impedance_correlation = impedance_comparison[
    correlation_columns
].corr(method="pearson")

impedance_correlation

In [ ]:
# Save impedance analysis results
impedance_comparison.to_csv(
    "B0005_impedance_analysis.csv",
    index=False
)

final_df.to_csv(
    "B0005_final_dataset.csv",
    index=False
)

print("Files saved successfully:")
print("1. B0005_impedance_analysis.csv")
print("2. B0005_final_dataset.csv")

In [ ]:
from scipy.stats import pearsonr, spearmanr

# Use only past-only impedance matches within 1 hour
model_data = final_df.dropna(
    subset=["Re", "Rct"]
).copy()

print("Valid observations:", len(model_data))

for feature in ["Re", "Rct"]:
    pearson_corr, pearson_p = pearsonr(
        model_data[feature],
        model_data["SoH_percent"]
    )

    spearman_corr, spearman_p = spearmanr(
        model_data[feature],
        model_data["SoH_percent"]
    )

    print(f"\n{feature}")
    print(f"Pearson:  {pearson_corr:.4f} (p={pearson_p:.4g})")
    print(f"Spearman: {spearman_corr:.4f} (p={spearman_p:.4g})")

In [ ]:
# Full dataset: all 168 discharge cycles
full_model_data = final_df.copy()

# Impedance dataset: only 132 valid past-only matches
impedance_model_data = final_df.dropna(
    subset=["Re", "Rct"]
).copy()

print("Full dataset:", full_model_data.shape)
print("Impedance dataset:", impedance_model_data.shape)

print("\nMissing values in impedance dataset:")
print(
    impedance_model_data[
        ["Re", "Rct"]
    ].isna().sum()
)

In [ ]:
# Define target
target = "SoH_percent"

# Model A: electrical and thermal features
features_A = [
    "voltage_mean",
    "voltage_min",
    "voltage_max",
    "current_mean",
    "temperature_mean",
    "temperature_max",
    "temperature_rise"
]

# Model B: electrical, thermal and impedance features
features_B = features_A + [
    "Re",
    "Rct"
]

# Use the same 132 observations for both experiments
X_A = impedance_model_data[features_A]
X_B = impedance_model_data[features_B]

y = impedance_model_data[target]

print("Model A features:", X_A.shape)
print("Model B features:", X_B.shape)
print("Target:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

# Chronological split: first 80% for training, last 20% for testing
split_index = int(len(impedance_model_data) * 0.8)

X_A_train = X_A.iloc[:split_index]
X_A_test = X_A.iloc[split_index:]

X_B_train = X_B.iloc[:split_index]
X_B_test = X_B.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training observations:", len(y_train))
print("Testing observations:", len(y_test))

print("\nTraining cycle range:")
print(
    impedance_model_data["record_index"].iloc[0],
    "to",
    impedance_model_data["record_index"].iloc[split_index - 1]
)

print("\nTesting cycle range:")
print(
    impedance_model_data["record_index"].iloc[split_index],
    "to",
    impedance_model_data["record_index"].iloc[-1]
)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Train Model A: voltage and temperature
model_A = LinearRegression()
model_A.fit(X_A_train, y_train)

# Train Model B: voltage, temperature and impedance
model_B = LinearRegression()
model_B.fit(X_B_train, y_train)

# Generate predictions
pred_A = model_A.predict(X_A_test)
pred_B = model_B.predict(X_B_test)

# Evaluation function
def evaluate_model(y_true, predictions, model_name):
    mae = mean_absolute_error(y_true, predictions)
    rmse = np.sqrt(mean_squared_error(y_true, predictions))
    r2 = r2_score(y_true, predictions)

    print(f"\n{model_name}")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²:   {r2:.4f}")

# Evaluate both models
evaluate_model(y_test, pred_A, "Model A — Voltage and Temperature")
evaluate_model(y_test, pred_B, "Model B — With Impedance")

In [ ]:
# Compare actual and predicted SoH
comparison_df = pd.DataFrame({
    "Cycle": impedance_model_data["record_index"].iloc[split_index:],
    "Actual_SoH": y_test.values,
    "Model_A_Predicted": pred_A,
    "Model_B_Predicted": pred_B
})

plt.figure(figsize=(12, 5))

plt.plot(
    comparison_df["Cycle"],
    comparison_df["Actual_SoH"],
    marker="o",
    label="Actual SoH"
)

plt.plot(
    comparison_df["Cycle"],
    comparison_df["Model_A_Predicted"],
    marker="s",
    label="Model A"
)

plt.plot(
    comparison_df["Cycle"],
    comparison_df["Model_B_Predicted"],
    marker="^",
    label="Model B"
)

plt.xlabel("Discharge Cycle")
plt.ylabel("SoH (%)")
plt.title("Actual vs Predicted Battery State of Health")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

comparison_df

In [ ]:
# Extract coefficients from both models
coefficients_df = pd.DataFrame({
    "Feature": features_B,
    "Model_A_Coefficient": [
        model_A.coef_[features_A.index(f)] if f in features_A else np.nan
        for f in features_B
    ],
    "Model_B_Coefficient": model_B.coef_
})

coefficients_df

In [ ]:
# Display coefficients visually
plt.figure(figsize=(11, 5))

plt.bar(
    coefficients_df["Feature"],
    coefficients_df["Model_B_Coefficient"]
)

plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Regression coefficient")
plt.title("Linear Regression — Model B Feature Coefficients")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

# Standardise predictors for numerical stability
scaler = StandardScaler()
X_B_scaled = scaler.fit_transform(X_B_train)

vif_df = pd.DataFrame({
    "Feature": features_B,
    "VIF": [
        variance_inflation_factor(X_B_scaled, i)
        for i in range(X_B_scaled.shape[1])
    ]
})

vif_df.sort_values("VIF", ascending=False)

In [ ]:
reduced_features = [
    "voltage_mean",
    "voltage_min",
    "temperature_rise",
    "Re",
    "Rct"
]

X_reduced = impedance_model_data[reduced_features]
X_reduced_train = X_reduced.iloc[:split_index]

scaler_reduced = StandardScaler()
X_reduced_scaled = scaler_reduced.fit_transform(
    X_reduced_train
)

reduced_vif_df = pd.DataFrame({
    "Feature": reduced_features,
    "VIF": [
        variance_inflation_factor(X_reduced_scaled, i)
        for i in range(X_reduced_scaled.shape[1])
    ]
})

reduced_vif_df.sort_values("VIF", ascending=False)

In [ ]:
diagnostic_features = [
    "voltage_min",
    "temperature_rise",
    "Re",
    "Rct"
]

X_diagnostic = impedance_model_data[
    diagnostic_features
].iloc[:split_index]

X_diagnostic_scaled = StandardScaler().fit_transform(
    X_diagnostic
)

diagnostic_vif = pd.DataFrame({
    "Feature": diagnostic_features,
    "VIF": [
        variance_inflation_factor(
            X_diagnostic_scaled, i
        )
        for i in range(len(diagnostic_features))
    ]
})

diagnostic_vif.sort_values("VIF", ascending=False)

In [ ]:
# Compare two alternative feature combinations
feature_sets = {
    "Without Re": [
        "voltage_min",
        "temperature_rise",
        "Rct"
    ],
    "Without Rct": [
        "voltage_min",
        "temperature_rise",
        "Re"
    ]
}

for name, selected_features in feature_sets.items():

    X_temp = impedance_model_data[
        selected_features
    ].iloc[:split_index]

    X_scaled = StandardScaler().fit_transform(X_temp)

    print(f"\n{name}")

    for i, feature in enumerate(selected_features):
        vif = variance_inflation_factor(X_scaled, i)
        print(f"{feature}: {vif:.3f}")

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Ridge A: Voltage and temperature
ridge_A = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

# Ridge B: Voltage, temperature and impedance
ridge_B = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

# Train
ridge_A.fit(X_A_train, y_train)
ridge_B.fit(X_B_train, y_train)

# Predict
ridge_pred_A = ridge_A.predict(X_A_test)
ridge_pred_B = ridge_B.predict(X_B_test)

# Evaluate
evaluate_model(
    y_test,
    ridge_pred_A,
    "Ridge Regression — Voltage and Temperature"
)

evaluate_model(
    y_test,
    ridge_pred_B,
    "Ridge Regression — With Impedance"
)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

# Tune Model A
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

param_grid = {
    "ridge__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}

tscv = TimeSeriesSplit(n_splits=5)

grid_A = GridSearchCV(
    ridge_pipeline,
    param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error"
)

grid_A.fit(X_A_train, y_train)

print("Best alpha:", grid_A.best_params_["ridge__alpha"])
print("Best validation MAE:", -grid_A.best_score_)

In [ ]:
# Retrieve the best Ridge model
best_ridge_A = grid_A.best_estimator_

# Predict on the untouched test set
tuned_ridge_pred_A = best_ridge_A.predict(X_A_test)

# Evaluate
evaluate_model(
    y_test,
    tuned_ridge_pred_A,
    "Optimised Ridge — Voltage and Temperature"
)

In [ ]:
fold_results = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_A_train), start=1):
    X_fold_train = X_A_train.iloc[train_idx]
    X_fold_val = X_A_train.iloc[val_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    )

    model.fit(X_fold_train, y_fold_train)
    predictions = model.predict(X_fold_val)

    fold_mae = mean_absolute_error(y_fold_val, predictions)

    fold_results.append({
        "Fold": fold,
        "Training samples": len(train_idx),
        "Validation samples": len(val_idx),
        "Validation MAE": fold_mae
    })

pd.DataFrame(fold_results)

In [ ]:
alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]

ridge_cv_results = []

for alpha in alpha_values:
    fold_maes = []

    for train_idx, val_idx in tscv.split(X_A_train):
        X_fold_train = X_A_train.iloc[train_idx]
        X_fold_val = X_A_train.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=alpha)
        )

        model.fit(X_fold_train, y_fold_train)
        predictions = model.predict(X_fold_val)

        fold_maes.append(
            mean_absolute_error(y_fold_val, predictions)
        )

    ridge_cv_results.append({
        "Alpha": alpha,
        "Fold 1 MAE": fold_maes[0],
        "Fold 2 MAE": fold_maes[1],
        "Fold 3 MAE": fold_maes[2],
        "Fold 4 MAE": fold_maes[3],
        "Fold 5 MAE": fold_maes[4],
        "Mean CV MAE": np.mean(fold_maes)
    })

ridge_cv_comparison = pd.DataFrame(ridge_cv_results)
ridge_cv_comparison

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rf_A = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

rf_A.fit(X_A_train, y_train)

rf_predictions_A = rf_A.predict(X_A_test)

evaluate_model(
    y_test,
    rf_predictions_A,
    "Random Forest — Voltage and Temperature"
)

In [ ]:
rf_comparison = pd.DataFrame({
    "Actual SoH": y_test.values,
    "Predicted SoH": rf_predictions_A
})

rf_comparison["Error"] = (
    rf_comparison["Predicted SoH"] -
    rf_comparison["Actual SoH"]
)

print(rf_comparison.head(10))

print("\nActual SoH range:")
print(y_train.min(), "to", y_train.max())

print("\nPredicted SoH range:")
print(rf_predictions_A.min(), "to", rf_predictions_A.max())

In [ ]:
rf_B = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

rf_B.fit(X_B_train, y_train)

rf_predictions_B = rf_B.predict(X_B_test)

evaluate_model(
    y_test,
    rf_predictions_B,
    "Random Forest — Voltage, Temperature and Impedance"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(
    impedance_model_data["Re"],
    impedance_model_data["SoH_percent"]
)
axes[0].set_xlabel("Electrolyte Resistance Re (Ω)")
axes[0].set_ylabel("State of Health (%)")
axes[0].set_title("Re vs SoH")

axes[1].scatter(
    impedance_model_data["Rct"],
    impedance_model_data["SoH_percent"]
)
axes[1].set_xlabel("Charge-transfer Resistance Rct (Ω)")
axes[1].set_ylabel("State of Health (%)")
axes[1].set_title("Rct vs SoH")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_physics = impedance_model_data[["Re", "Rct"]]
y_physics = impedance_model_data["SoH_percent"]

X_physics_train = X_physics.iloc[:split_index]
X_physics_test = X_physics.iloc[split_index:]

y_physics_train = y_physics.iloc[:split_index]
y_physics_test = y_physics.iloc[split_index:]

physics_model = LinearRegression()

physics_model.fit(X_physics_train, y_physics_train)

physics_predictions = physics_model.predict(X_physics_test)

evaluate_model(
    y_physics_test,
    physics_predictions,
    "Resistance-Based Model — Re and Rct"
)

In [ ]:
physics_comparison = pd.DataFrame({
    "Actual SoH": y_physics_test.values,
    "Predicted SoH": physics_predictions
})

physics_comparison["Error"] = (
    physics_comparison["Predicted SoH"] -
    physics_comparison["Actual SoH"]
)

print(physics_comparison.head(10))

print("\nTraining feature ranges:")
print(X_physics_train.describe().loc[["min", "max"]])

print("\nTesting feature ranges:")
print(X_physics_test.describe().loc[["min", "max"]])

print("\nPredicted SoH range:")
print(
    physics_predictions.min(),
    "to",
    physics_predictions.max()
)

In [ ]:
print("TRAINING DATA")
print(
    impedance_model_data.iloc[:split_index][
        ["Re", "Rct", "SoH_percent"]
    ].corr()
)

print("\nTEST DATA")
print(
    impedance_model_data.iloc[split_index:][
        ["Re", "Rct", "SoH_percent"]
    ].corr()
)

print("\nMEAN VALUES")

print(
    impedance_model_data.iloc[:split_index][
        ["Re", "Rct", "SoH_percent"]
    ].mean()
)

print(
    impedance_model_data.iloc[split_index:][
        ["Re", "Rct", "SoH_percent"]
    ].mean()
)

In [ ]:
X_rct = impedance_model_data[["Rct"]]

X_rct_train = X_rct.iloc[:split_index]
X_rct_test = X_rct.iloc[split_index:]

rct_model = LinearRegression()

rct_model.fit(X_rct_train, y_physics_train)

rct_predictions = rct_model.predict(X_rct_test)

evaluate_model(
    y_physics_test,
    rct_predictions,
    "Resistance-Based Model — Rct Only"
)

In [ ]:
print("Rct model coefficient:", rct_model.coef_[0])
print("Rct model intercept:", rct_model.intercept_)

print("\nTraining Rct range:")
print(X_rct_train["Rct"].min(), X_rct_train["Rct"].max())

print("\nTest Rct range:")
print(X_rct_test["Rct"].min(), X_rct_test["Rct"].max())

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    X_rct_train["Rct"],
    y_physics_train,
    label="Training data",
    alpha=0.7
)

plt.scatter(
    X_rct_test["Rct"],
    y_physics_test,
    label="Test data",
    alpha=0.7
)

rct_range = np.linspace(
    X_rct_train["Rct"].min(),
    X_rct_train["Rct"].max(),
    100
)

plt.plot(
    rct_range,
    rct_model.predict(pd.DataFrame({"Rct": rct_range})),
    label="Training regression line"
)

plt.xlabel("Charge-transfer Resistance Rct (Ω)")
plt.ylabel("State of Health (%)")
plt.title("Rct-Based SoH Model: Training vs Test")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
features_C = [
    "voltage_min",
    "temperature_rise",
    "Re",
    "Rct"
]

X_C = impedance_model_data[features_C]

X_C_train = X_C.iloc[:split_index]
X_C_test = X_C.iloc[split_index:]

ridge_C = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

ridge_C.fit(X_C_train, y_train)

ridge_predictions_C = ridge_C.predict(X_C_test)

evaluate_model(
    y_test,
    ridge_predictions_C,
    "Ridge — Reduced Physical and Electrical Features"
)

In [ ]:
ridge_C_comparison = pd.DataFrame({
    "Actual SoH": y_test.values,
    "Predicted SoH": ridge_predictions_C
})

ridge_C_comparison["Error"] = (
    ridge_C_comparison["Predicted SoH"] -
    ridge_C_comparison["Actual SoH"]
)

print(ridge_C_comparison.head(10))

print("\nActual SoH range:")
print(y_test.min(), "to", y_test.max())

print("\nPredicted SoH range:")
print(
    ridge_predictions_C.min(),
    "to",
    ridge_predictions_C.max()
)

In [ ]:
persistence_predictions = np.full(
    len(y_test),
    y_train.iloc[-1]
)

evaluate_model(
    y_test,
    persistence_predictions,
    "Persistence Baseline — Last Training SoH"
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    range(len(y_test)),
    y_test.values,
    marker="o",
    label="Actual SoH"
)

plt.plot(
    range(len(y_test)),
    ridge_A.predict(X_A_test),
    marker="s",
    label="Predicted SoH — Ridge"
)

plt.xlabel("Test observation")
plt.ylabel("State of Health (%)")
plt.title("Ridge Model: Actual vs Predicted Battery SoH")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
model_results = pd.DataFrame([
    {
        "Model": "Ridge — Voltage & Temperature",
        "MAE": 0.3620,
        "RMSE": 0.4093,
        "R2": 0.8259
    },
    {
        "Model": "Ridge — Tuned",
        "MAE": 0.5445,
        "RMSE": 0.6099,
        "R2": 0.6133
    },
    {
        "Model": "Ridge — Voltage, Temperature & Impedance",
        "MAE": 0.7342,
        "RMSE": 0.7624,
        "R2": 0.3959
    },
    {
        "Model": "Random Forest — Voltage & Temperature",
        "MAE": 2.4350,
        "RMSE": 2.6172,
        "R2": -6.1192
    },
    {
        "Model": "Random Forest — With Impedance",
        "MAE": 2.5105,
        "RMSE": 2.6886,
        "R2": -6.5130
    },
    {
        "Model": "Resistance — Re & Rct",
        "MAE": 5.8278,
        "RMSE": 6.3595,
        "R2": -41.0343
    },
    {
        "Model": "Persistence Baseline",
        "MAE": 1.9246,
        "RMSE": 2.1599,
        "R2": -3.8487
    }
])

model_results.sort_values("MAE")

In [ ]:
model_results.to_csv(
    "B0005_model_comparison.csv",
    index=False
)

print("Model comparison saved successfully.")

In [ ]:
import os
import glob

mat_files = glob.glob("*.mat")

print("Available battery files:")

for file in sorted(mat_files):
    print(f"{file} — {os.path.getsize(file) / 1024:.1f} KB")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import glob
import os

mat_files = sorted(glob.glob("*.mat"))

print("Available battery files:")
for file in mat_files:
    print(f"{file} — {os.path.getsize(file) / 1024:.1f} KB")

In [ ]:
from scipy.io import loadmat
import numpy as np

for battery_id in ["B0006", "B0007", "B0018"]:
    mat = loadmat(
        f"{battery_id}.mat",
        squeeze_me=True,
        struct_as_record=False
    )

    battery = mat[battery_id]
    cycles = np.atleast_1d(battery.cycle)

    discharge_count = sum(
        1 for cycle in cycles
        if cycle.type == "discharge"
    )

    impedance_count = sum(
        1 for cycle in cycles
        if cycle.type == "impedance"
    )

    print(f"\n{battery_id}")
    print("Total records:", len(cycles))
    print("Discharge records:", discharge_count)
    print("Impedance records:", impedance_count)

In [ ]:
mat = loadmat(
    "B0018.mat",
    squeeze_me=True,
    struct_as_record=False
)

battery = mat["B0018"]
cycles = np.atleast_1d(battery.cycle)

discharge = next(c for c in cycles if c.type == "discharge")
impedance = next(c for c in cycles if c.type == "impedance")

print("Discharge fields:")
print(discharge.data._fieldnames)

print("\nImpedance fields:")
print(impedance.data._fieldnames)

In [ ]:
import numpy as np
import pandas as pd
from scipy.io import loadmat


def extract_discharge_features(battery_id):

    # 1. Load MATLAB file
    mat = loadmat(
        f"{battery_id}.mat",
        squeeze_me=True,
        struct_as_record=False
    )

    battery = mat[battery_id]
    cycles = np.atleast_1d(battery.cycle)

    rows = []

    # 2. Extract discharge cycles
    for record_index, cycle in enumerate(cycles):

        if cycle.type != "discharge":
            continue

        data = cycle.data

        voltage = np.asarray(data.Voltage_measured).ravel()
        current = np.asarray(data.Current_measured).ravel()
        temperature = np.asarray(data.Temperature_measured).ravel()
        time = np.asarray(data.Time).ravel()

        capacity = float(data.Capacity)

        # 3. Convert MATLAB date into pandas Timestamp
        date_components = np.asarray(cycle.time, dtype=float).ravel()

        seconds = date_components[5]
        whole_seconds = int(np.floor(seconds))
        fractional_seconds = seconds - whole_seconds

        timestamp = pd.Timestamp(
            year=int(date_components[0]),
            month=int(date_components[1]),
            day=int(date_components[2]),
            hour=int(date_components[3]),
            minute=int(date_components[4]),
            second=whole_seconds
        ) + pd.to_timedelta(fractional_seconds, unit="s")

        # 4. Extract cycle-level features
        rows.append({

            "battery_id": battery_id,
            "record_index": record_index,
            "timestamp": timestamp,

            "capacity_Ah": capacity,

            "voltage_mean": np.mean(voltage),
            "voltage_min": np.min(voltage),
            "voltage_max": np.max(voltage),

            "current_mean": np.mean(current),

            "temperature_mean": np.mean(temperature),
            "temperature_min": np.min(temperature),
            "temperature_max": np.max(temperature),

            "temperature_rise": np.max(temperature) - np.min(temperature),

            "discharge_duration_s": time[-1] - time[0]
        })

    # 5. Create DataFrame
    df = pd.DataFrame(rows)

    # 6. Calculate SoH using the battery's initial measured capacity
    initial_capacity = df["capacity_Ah"].iloc[0]

    df["SoH_percent"] = (
        df["capacity_Ah"] / initial_capacity
    ) * 100

    return df

In [ ]:
test_df = extract_discharge_features("B0005")

print("Dataset shape:", test_df.shape)
display(test_df.head())

print("\nInitial capacity:", test_df["capacity_Ah"].iloc[0])
print("Initial SoH:", test_df["SoH_percent"].iloc[0])
print("Final SoH:", test_df["SoH_percent"].iloc[-1])

In [ ]:
battery_ids = ["B0005", "B0006", "B0007", "B0018"]

battery_datasets = {}

for battery_id in battery_ids:

    df = extract_discharge_features(battery_id)

    battery_datasets[battery_id] = df

    print(f"\n{battery_id}")
    print("Number of cycles:", len(df))
    print("Initial capacity (Ah):", round(df["capacity_Ah"].iloc[0], 4))
    print("Final capacity (Ah):", round(df["capacity_Ah"].iloc[-1], 4))
    print("Final SoH (%):", round(df["SoH_percent"].iloc[-1], 2))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11, 6))

for battery_id, df in battery_datasets.items():

    plt.plot(
        range(1, len(df) + 1),
        df["SoH_percent"],
        label=battery_id
    )

plt.xlabel("Discharge Cycle")
plt.ylabel("State of Health (SoH %)")
plt.title("Capacity Degradation Across Four NASA Batteries")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def extract_impedance_features(battery_id):

    mat = loadmat(
        f"{battery_id}.mat",
        squeeze_me=True,
        struct_as_record=False
    )

    battery = mat[battery_id]
    cycles = np.atleast_1d(battery.cycle)

    rows = []

    for record_index, cycle in enumerate(cycles):

        if cycle.type != "impedance":
            continue

        data = cycle.data

        date_components = np.asarray(
            cycle.time, dtype=float
        ).ravel()

        seconds = date_components[5]
        whole_seconds = int(np.floor(seconds))
        fractional_seconds = seconds - whole_seconds

        timestamp = pd.Timestamp(
            year=int(date_components[0]),
            month=int(date_components[1]),
            day=int(date_components[2]),
            hour=int(date_components[3]),
            minute=int(date_components[4]),
            second=whole_seconds
        ) + pd.to_timedelta(fractional_seconds, unit="s")

        rows.append({

            "battery_id": battery_id,
            "impedance_record_index": record_index,
            "impedance_timestamp": timestamp,

            "Re": float(data.Re),
            "Rct": float(data.Rct)
        })

    return pd.DataFrame(rows)

In [ ]:
impedance_test = extract_impedance_features("B0005")

print("Dataset shape:", impedance_test.shape)
display(impedance_test.head())

print("\nRe statistics:")
print(impedance_test["Re"].describe())

print("\nRct statistics:")
print(impedance_test["Rct"].describe())

In [ ]:
def match_impedance_to_discharge(
    discharge_df,
    impedance_df,
    tolerance="1h"
):

    # Sort both datasets by timestamp
    discharge_sorted = discharge_df.sort_values(
        "timestamp"
    ).copy()

    impedance_sorted = impedance_df.sort_values(
        "impedance_timestamp"
    ).copy()

    # Match each discharge with its latest preceding
    # impedance measurement within the specified tolerance
    matched_df = pd.merge_asof(
        discharge_sorted,
        impedance_sorted[
            [
                "impedance_timestamp",
                "impedance_record_index",
                "Re",
                "Rct"
            ]
        ],
        left_on="timestamp",
        right_on="impedance_timestamp",
        direction="backward",
        tolerance=pd.Timedelta(tolerance)
    )

    # Calculate the age of the impedance measurement
    matched_df["impedance_age_hours"] = (
        matched_df["timestamp"]
        - matched_df["impedance_timestamp"]
    ).dt.total_seconds() / 3600

    return matched_df

In [ ]:
matched_test = match_impedance_to_discharge(
    battery_datasets["B0005"],
    impedance_test
)

print("Total discharge cycles:", len(matched_test))

print(
    "Cycles with impedance:",
    matched_test["Re"].notna().sum()
)

print(
    "Cycles without impedance:",
    matched_test["Re"].isna().sum()
)

print("\nImpedance age statistics (hours):")
print(matched_test["impedance_age_hours"].describe())

display(matched_test.head(10))

In [ ]:
impedance_datasets = {}
matched_datasets = {}

for battery_id in battery_ids:

    # Extract impedance measurements
    impedance_df = extract_impedance_features(battery_id)

    # Match them with discharge measurements
    matched_df = match_impedance_to_discharge(
        battery_datasets[battery_id],
        impedance_df,
        tolerance="1h"
    )

    impedance_datasets[battery_id] = impedance_df
    matched_datasets[battery_id] = matched_df

    print(f"\n{battery_id}")
    print("Discharge cycles:", len(matched_df))
    print("Impedance records:", len(impedance_df))
    print("Matched cycles:", matched_df["Re"].notna().sum())
    print("Unmatched cycles:", matched_df["Re"].isna().sum())
    print(
        "Matching percentage:",
        round(matched_df["Re"].notna().mean() * 100, 2),
        "%"
    )

In [ ]:
# Combine all four matched datasets
combined_df = pd.concat(
    matched_datasets.values(),
    ignore_index=True
)

# Add an explicit impedance availability indicator
combined_df["impedance_available"] = (
    combined_df["Re"].notna()
).astype(int)

# Sort by battery and timestamp
combined_df = combined_df.sort_values(
    ["battery_id", "timestamp"]
).reset_index(drop=True)

print("Combined dataset shape:", combined_df.shape)

print("\nObservations per battery:")
print(combined_df.groupby("battery_id").size())

print("\nImpedance availability:")
print(
    combined_df.groupby("battery_id")["impedance_available"]
    .agg(["sum", "count", "mean"])
)

display(combined_df.head())

In [ ]:
# Compare SoH for cycles with and without impedance measurements

availability_analysis = (
    combined_df
    .groupby(["battery_id", "impedance_available"])["SoH_percent"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

print("SoH distribution by impedance availability:")
display(availability_analysis)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, battery_id in zip(axes.flatten(), battery_ids):

    battery_data = combined_df[
        combined_df["battery_id"] == battery_id
    ]

    matched = battery_data.loc[
        battery_data["impedance_available"] == 1,
        "SoH_percent"
    ]

    unmatched = battery_data.loc[
        battery_data["impedance_available"] == 0,
        "SoH_percent"
    ]

    ax.boxplot(
        [matched, unmatched],
        tick_labels=["Available", "Missing"]
    )

    ax.set_title(battery_id)
    ax.set_ylabel("SoH (%)")
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("SoH Distribution by Impedance Availability")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=False)

for ax, battery_id in zip(axes, battery_ids):

    df = combined_df[
        combined_df["battery_id"] == battery_id
    ].copy()

    ax.scatter(
        range(1, len(df) + 1),
        df["impedance_available"],
        s=18
    )

    ax.set_title(battery_id)
    ax.set_ylabel("Impedance")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Missing", "Available"])
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Discharge Cycle")

plt.suptitle("Impedance Measurement Availability Across Battery Life")
plt.tight_layout()
plt.show()

In [ ]:
# Dataset A: all discharge cycles
model_dataset_A = combined_df.copy()

# Dataset B: only cycles with valid impedance measurements
model_dataset_B = combined_df[
    combined_df["impedance_available"] == 1
].copy()

# Reset indices
model_dataset_A = model_dataset_A.reset_index(drop=True)
model_dataset_B = model_dataset_B.reset_index(drop=True)

# Verify dataset sizes
print("Dataset A — Voltage and Temperature")
print("Shape:", model_dataset_A.shape)

print("\nDataset B — Voltage, Temperature and Impedance")
print("Shape:", model_dataset_B.shape)

print("\nObservations per battery in Dataset B:")
print(model_dataset_B.groupby("battery_id").size())

# Verify no missing impedance values in Dataset B
print(
    "\nMissing Re values:",
    model_dataset_B["Re"].isna().sum()
)

print(
    "Missing Rct values:",
    model_dataset_B["Rct"].isna().sum()
)

In [ ]:
# Define the input features

features_A = [
    "voltage_mean",
    "voltage_min",
    "voltage_max",
    "current_mean",
    "temperature_mean",
    "temperature_max",
    "temperature_rise"
]

features_B = features_A + ["Re", "Rct"]


# Prepare Leave-One-Battery-Out datasets

loso_datasets = {}

for test_battery in battery_ids:

    # Dataset A: all discharge observations from other batteries
    train_A = model_dataset_A[
        model_dataset_A["battery_id"] != test_battery
    ].copy()

    # Dataset B: impedance-available observations from other batteries
    train_B = model_dataset_B[
        model_dataset_B["battery_id"] != test_battery
    ].copy()

    # Use the same matched test observations for both models
    test_data = model_dataset_B[
        model_dataset_B["battery_id"] == test_battery
    ].copy()

    loso_datasets[test_battery] = {
        "train_A": train_A,
        "train_B": train_B,
        "test": test_data
    }

    print(f"\nTest battery: {test_battery}")
    print("Dataset A training observations:", len(train_A))
    print("Dataset B training observations:", len(train_B))
    print("Common test observations:", len(test_data))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

results = []

for test_battery in battery_ids:

    datasets = loso_datasets[test_battery]

    # Training data
    train_A = datasets["train_A"]
    train_B = datasets["train_B"]

    # Common test data
    test_data = datasets["test"]

    y_train_A = train_A["SoH_percent"]
    y_train_B = train_B["SoH_percent"]
    y_test = test_data["SoH_percent"]

    # Define models
    model_A = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1))
    ])

    model_B = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1))
    ])

    # Train Model A: voltage and temperature
    model_A.fit(train_A[features_A], y_train_A)

    # Train Model B: voltage, temperature and impedance
    model_B.fit(train_B[features_B], y_train_B)

    # Predictions on the same held-out battery observations
    predictions_A = model_A.predict(test_data[features_A])
    predictions_B = model_B.predict(test_data[features_B])

    # Calculate evaluation metrics
    for model_name, predictions in [
        ("Ridge - Voltage & Temperature", predictions_A),
        ("Ridge - With Impedance", predictions_B)
    ]:

        results.append({
            "test_battery": test_battery,
            "model": model_name,
            "observations": len(y_test),
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": np.sqrt(
                mean_squared_error(y_test, predictions)
            ),
            "R2": r2_score(y_test, predictions)
        })

# Create results table
loso_results = pd.DataFrame(results)

print("Leave-One-Battery-Out Results:")
display(
    loso_results.round(4)
)

In [ ]:
results_fair = []

for test_battery in battery_ids:

    # Use the same impedance-available training observations
    # for both models
    train_data = model_dataset_B[
        model_dataset_B["battery_id"] != test_battery
    ].copy()

    # Same held-out test observations
    test_data = model_dataset_B[
        model_dataset_B["battery_id"] == test_battery
    ].copy()

    y_train = train_data["SoH_percent"]
    y_test = test_data["SoH_percent"]

    # Model A: voltage and temperature
    model_A = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1))
    ])

    # Model B: voltage, temperature and impedance
    model_B = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1))
    ])

    model_A.fit(train_data[features_A], y_train)
    model_B.fit(train_data[features_B], y_train)

    predictions_A = model_A.predict(test_data[features_A])
    predictions_B = model_B.predict(test_data[features_B])

    for model_name, predictions in [
        ("Voltage & Temperature", predictions_A),
        ("Voltage, Temperature & Impedance", predictions_B)
    ]:

        results_fair.append({
            "test_battery": test_battery,
            "model": model_name,
            "observations": len(y_test),
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": np.sqrt(
                mean_squared_error(y_test, predictions)
            ),
            "R2": r2_score(y_test, predictions)
        })

fair_results_df = pd.DataFrame(results_fair)

display(fair_results_df.round(4))

In [ ]:
prediction_results = {}

for test_battery in battery_ids:

    train_data = model_dataset_B[
        model_dataset_B["battery_id"] != test_battery
    ].copy()

    test_data = model_dataset_B[
        model_dataset_B["battery_id"] == test_battery
    ].copy()

    y_train = train_data["SoH_percent"]
    y_test = test_data["SoH_percent"]

    # Model A: voltage and temperature
    model_A = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1))
    ])

    # Model B: voltage, temperature and impedance
    model_B = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1))
    ])

    model_A.fit(train_data[features_A], y_train)
    model_B.fit(train_data[features_B], y_train)

    predictions_A = model_A.predict(test_data[features_A])
    predictions_B = model_B.predict(test_data[features_B])

    prediction_results[test_battery] = pd.DataFrame({
        "actual_SoH": y_test.to_numpy(),
        "predicted_A": predictions_A,
        "predicted_B": predictions_B
    }).reset_index(drop=True)


# Plot predictions for each battery
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, battery_id in zip(axes.flatten(), battery_ids):

    results = prediction_results[battery_id]

    ax.plot(
        results["actual_SoH"].values,
        label="Actual SoH",
        linewidth=2
    )

    ax.plot(
        results["predicted_A"].values,
        label="Voltage & Temperature",
        linestyle="--"
    )

    ax.plot(
        results["predicted_B"].values,
        label="With Impedance",
        linestyle=":"
    )

    ax.set_title(battery_id)
    ax.set_xlabel("Test Observation")
    ax.set_ylabel("SoH (%)")
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Cross-Battery SoH Estimation: Actual vs Predicted")
plt.tight_layout()
plt.show()

In [ ]:
bias_results = []

for battery_id in battery_ids:

    results = prediction_results[battery_id]

    for model_name, prediction_column in [
        ("Voltage & Temperature", "predicted_A"),
        ("With Impedance", "predicted_B")
    ]:

        errors = (
            results[prediction_column] -
            results["actual_SoH"]
        )

        bias_results.append({
            "battery": battery_id,
            "model": model_name,
            "mean_error": errors.mean(),
            "median_error": errors.median(),
            "mean_absolute_error": errors.abs().mean(),
            "maximum_absolute_error": errors.abs().max()
        })

bias_df = pd.DataFrame(bias_results)

display(bias_df.round(3))

In [ ]:
baseline_results = []

for test_battery in battery_ids:

    # Same training and test population as the fair experiment
    train_data = model_dataset_B[
        model_dataset_B["battery_id"] != test_battery
    ].copy()

    test_data = model_dataset_B[
        model_dataset_B["battery_id"] == test_battery
    ].copy()

    y_test = test_data["SoH_percent"]

    # Baseline: predict the training-set mean SoH
    baseline_prediction = np.full(
        len(test_data),
        train_data["SoH_percent"].mean()
    )

    # Evaluate baseline
    baseline_results.append({
        "battery": test_battery,
        "training_mean_SoH": train_data["SoH_percent"].mean(),
        "baseline_MAE": mean_absolute_error(
            y_test, baseline_prediction
        ),
        "baseline_RMSE": np.sqrt(
            mean_squared_error(y_test, baseline_prediction)
        ),
        "baseline_R2": r2_score(
            y_test, baseline_prediction
        )
    })

baseline_df = pd.DataFrame(baseline_results)

display(baseline_df.round(4))

In [ ]:
# Merge baseline and Ridge results
comparison_df = fair_results_df.merge(
    baseline_df[["battery", "baseline_MAE"]],
    left_on="test_battery",
    right_on="battery"
)

# Calculate percentage improvement
comparison_df["improvement_vs_baseline_%"] = (
    (comparison_df["baseline_MAE"] - comparison_df["MAE"])
    / comparison_df["baseline_MAE"]
) * 100

# Display relevant columns
display(
    comparison_df[
        [
            "test_battery",
            "model",
            "baseline_MAE",
            "MAE",
            "improvement_vs_baseline_%"
        ]
    ].round(2)
)

# Calculate observation-weighted overall MAE
print("\nOverall weighted MAE:")

for model_name, group in comparison_df.groupby("model"):

    weighted_mae = np.average(
        group["MAE"],
        weights=group["observations"]
    )

    print(f"{model_name}: {weighted_mae:.3f}")

weighted_baseline = np.average(
    baseline_df["baseline_MAE"],
    weights=[132, 132, 132, 23]
)

print(f"Baseline: {weighted_baseline:.3f}")

In [ ]:
from sklearn.inspection import permutation_importance

importance_results = []

for test_battery in battery_ids:

    train_data = model_dataset_B[
        model_dataset_B["battery_id"] != test_battery
    ].copy()

    test_data = model_dataset_B[
        model_dataset_B["battery_id"] == test_battery
    ].copy()

    model_B = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1))
    ])

    model_B.fit(
        train_data[features_B],
        train_data["SoH_percent"]
    )

    # Permutation importance on the held-out battery
    importance = permutation_importance(
        model_B,
        test_data[features_B],
        test_data["SoH_percent"],
        scoring="neg_mean_absolute_error",
        n_repeats=30,
        random_state=42
    )

    for feature, mean_importance, std_importance in zip(
        features_B,
        importance.importances_mean,
        importance.importances_std
    ):

        importance_results.append({
            "battery": test_battery,
            "feature": feature,
            "importance": mean_importance,
            "std": std_importance
        })

importance_df = pd.DataFrame(importance_results)

# Display results
display(
    importance_df.pivot(
        index="feature",
        columns="battery",
        values="importance"
    ).round(4)
)

In [ ]:
ablation_results = []

feature_sets = {
    "Voltage & Temperature": features_A,
    "Voltage & Temperature + Re": features_A + ["Re"],
    "Voltage & Temperature + Rct": features_A + ["Rct"],
    "Voltage & Temperature + Re + Rct": features_B
}

for test_battery in battery_ids:

    train_data = model_dataset_B[
        model_dataset_B["battery_id"] != test_battery
    ].copy()

    test_data = model_dataset_B[
        model_dataset_B["battery_id"] == test_battery
    ].copy()

    y_train = train_data["SoH_percent"]
    y_test = test_data["SoH_percent"]

    for model_name, selected_features in feature_sets.items():

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=1))
        ])

        model.fit(
            train_data[selected_features],
            y_train
        )

        predictions = model.predict(
            test_data[selected_features]
        )

        ablation_results.append({
            "battery": test_battery,
            "model": model_name,
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": np.sqrt(
                mean_squared_error(y_test, predictions)
            ),
            "R2": r2_score(y_test, predictions)
        })

ablation_df = pd.DataFrame(ablation_results)

display(
    ablation_df.pivot(
        index="model",
        columns="battery",
        values="MAE"
    ).round(3)
)

In [ ]:
# Observation-weighted MAE across all held-out batteries

summary_ablation = (
    ablation_df
    .groupby("model")
    .apply(
        lambda group: np.average(
            group["MAE"],
            weights=[
                132 if b != "B0018" else 23
                for b in group["battery"]
            ]
        ),
        include_groups=False
    )
    .reset_index(name="weighted_MAE")
    .sort_values("weighted_MAE")
)

display(summary_ablation.round(4))

In [ ]:
re_correlation_results = []

for battery_id in battery_ids:

    df = model_dataset_B[
        model_dataset_B["battery_id"] == battery_id
    ]

    pearson_corr = df["Re"].corr(
        df["SoH_percent"],
        method="pearson"
    )

    spearman_corr = df["Re"].corr(
        df["SoH_percent"],
        method="spearman"
    )

    re_correlation_results.append({
        "battery": battery_id,
        "observations": len(df),
        "Pearson_Re_SoH": pearson_corr,
        "Spearman_Re_SoH": spearman_corr
    })

re_correlation_df = pd.DataFrame(
    re_correlation_results
)

display(re_correlation_df.round(4))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, battery_id in zip(axes.flatten(), battery_ids):

    df = model_dataset_B[
        model_dataset_B["battery_id"] == battery_id
    ]

    ax.scatter(
        df["Re"],
        df["SoH_percent"],
        alpha=0.7
    )

    ax.set_title(battery_id)
    ax.set_xlabel("Electrolyte resistance Re (Ohms)")
    ax.set_ylabel("State of Health (%)")
    ax.grid(True, alpha=0.3)

plt.suptitle(
    "Relationship between Electrolyte Resistance and Battery SoH",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, battery_id in zip(axes.flatten(), battery_ids):

    df = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
    )

    ax2 = ax.twinx()

    line1 = ax.plot(
        df["record_index"],
        df["SoH_percent"],
        marker="o",
        markersize=3,
        label="SoH (%)"
    )

    line2 = ax2.plot(
        df["record_index"],
        df["Re"],
        marker="s",
        markersize=3,
        linestyle="--",
        label="Re (Ohms)"
    )

    ax.set_title(battery_id)
    ax.set_xlabel("Discharge cycle index")
    ax.set_ylabel("SoH (%)")
    ax2.set_ylabel("Re (Ohms)")

    lines = line1 + line2
    ax.legend(lines, [line.get_label() for line in lines],
              loc="best")

    ax.grid(True, alpha=0.3)

plt.suptitle(
    "Evolution of Battery Health and Electrolyte Resistance",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import pearsonr

adjusted_correlations_corrected = []

for battery_id in battery_ids:

    # Select impedance-matched observations
    df = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    # Use the original discharge-cycle indices
    cycle_number = df["record_index"].to_numpy()

    soh = df["SoH_percent"].to_numpy()
    re = df["Re"].to_numpy()

    # Fit linear trends against the actual cycle indices
    soh_trend = np.polyfit(cycle_number, soh, 1)
    re_trend = np.polyfit(cycle_number, re, 1)

    # Calculate residuals (variations not explained by cycle number)
    soh_residuals = (
        soh - np.polyval(soh_trend, cycle_number)
    )

    re_residuals = (
        re - np.polyval(re_trend, cycle_number)
    )

    # Calculate correlation between residuals
    corr, p_value = pearsonr(
        soh_residuals,
        re_residuals
    )

    adjusted_correlations_corrected.append({
        "battery": battery_id,
        "observations": len(df),
        "adjusted_correlation": corr,
        "p_value": p_value
    })

# Display results
adjusted_corr_corrected_df = pd.DataFrame(
    adjusted_correlations_corrected
)

display(adjusted_corr_corrected_df.round(4))

In [ ]:
import statsmodels.api as sm

regression_results = []

for battery_id in battery_ids:

    df = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    y = df["SoH_percent"]

    # Model 1: Cycle number only
    X1 = sm.add_constant(
        df[["record_index"]]
    )
    model_1 = sm.OLS(y, X1).fit()

    # Model 2: Cycle number + Re
    X2 = sm.add_constant(
        df[["record_index", "Re"]]
    )
    model_2 = sm.OLS(y, X2).fit()

    regression_results.append({
        "battery": battery_id,
        "observations": len(df),
        "R2_cycle_only": model_1.rsquared,
        "R2_cycle_plus_Re": model_2.rsquared,
        "R2_improvement": (
            model_2.rsquared - model_1.rsquared
        ),
        "Re_coefficient": model_2.params["Re"],
        "Re_p_value": model_2.pvalues["Re"]
    })

regression_df = pd.DataFrame(regression_results)

display(regression_df.round(4))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, battery_id in zip(axes.flatten(), battery_ids):

    df = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    X = sm.add_constant(
        df[["record_index", "Re"]]
    )

    y = df["SoH_percent"]

    model = sm.OLS(y, X).fit()

    residuals = model.resid

    ax.scatter(
        df["record_index"],
        residuals,
        alpha=0.7
    )

    ax.axhline(0, linestyle="--")

    ax.set_title(battery_id)
    ax.set_xlabel("Discharge cycle index")
    ax.set_ylabel("Regression residuals (SoH percentage points)")
    ax.grid(True, alpha=0.3)

plt.suptitle(
    "Residual Analysis: Cycle Number + Re Regression",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

nonlinear_results = []

for battery_id in battery_ids:

    df = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    X = df[["record_index", "Re"]]
    y = df["SoH_percent"]

    # Linear model
    linear_model = LinearRegression()
    linear_model.fit(X, y)

    pred_linear = linear_model.predict(X)

    # Polynomial model (degree 2)
    polynomial_model = Pipeline([
        ("poly", PolynomialFeatures(
            degree=2,
            include_bias=False
        )),
        ("regression", LinearRegression())
    ])

    polynomial_model.fit(X, y)

    pred_polynomial = polynomial_model.predict(X)

    nonlinear_results.append({
        "battery": battery_id,
        "Linear_MAE": mean_absolute_error(y, pred_linear),
        "Polynomial_MAE": mean_absolute_error(y, pred_polynomial),
        "Linear_R2": r2_score(y, pred_linear),
        "Polynomial_R2": r2_score(y, pred_polynomial)
    })

nonlinear_df = pd.DataFrame(nonlinear_results)

display(nonlinear_df.round(4))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

temporal_results = []

for battery_id in battery_ids:

    df = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    # Chronological 70/30 split
    split_index = int(len(df) * 0.70)

    train = df.iloc[:split_index]
    test = df.iloc[split_index:]

    X_train = train[["record_index", "Re"]]
    X_test = test[["record_index", "Re"]]

    y_train = train["SoH_percent"]
    y_test = test["SoH_percent"]

    # Linear model
    linear_model = LinearRegression()
    linear_model.fit(X_train, y_train)

    pred_linear = linear_model.predict(X_test)

    # Polynomial model
    polynomial_model = Pipeline([
        ("poly", PolynomialFeatures(
            degree=2,
            include_bias=False
        )),
        ("regression", LinearRegression())
    ])

    polynomial_model.fit(X_train, y_train)

    pred_polynomial = polynomial_model.predict(X_test)

    temporal_results.append({
        "battery": battery_id,
        "train_observations": len(train),
        "test_observations": len(test),
        "Linear_MAE": mean_absolute_error(
            y_test, pred_linear
        ),
        "Polynomial_MAE": mean_absolute_error(
            y_test, pred_polynomial
        ),
        "Linear_R2": r2_score(
            y_test, pred_linear
        ),
        "Polynomial_R2": r2_score(
            y_test, pred_polynomial
        )
    })

temporal_df = pd.DataFrame(temporal_results)

display(temporal_df.round(4))

In [ ]:
distribution_results = []

for battery_id in battery_ids:

    df = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    split_index = int(len(df) * 0.70)

    train = df.iloc[:split_index]
    test = df.iloc[split_index:]

    distribution_results.append({
        "battery": battery_id,
        "train_SoH_min": train["SoH_percent"].min(),
        "train_SoH_max": train["SoH_percent"].max(),
        "test_SoH_min": test["SoH_percent"].min(),
        "test_SoH_max": test["SoH_percent"].max()
    })

distribution_df = pd.DataFrame(distribution_results)

display(distribution_df.round(2))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

comparison_results = []

for battery_id in battery_ids:

    df = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    split_index = int(len(df) * 0.70)

    train = df.iloc[:split_index]
    test = df.iloc[split_index:]

    y_train = train["SoH_percent"]
    y_test = test["SoH_percent"]

    # Model 1: Cycle only
    X1_train = train[["record_index"]]
    X1_test = test[["record_index"]]

    model_1 = LinearRegression()
    model_1.fit(X1_train, y_train)
    pred_1 = model_1.predict(X1_test)

    # Model 2: Cycle + Re
    X2_train = train[["record_index", "Re"]]
    X2_test = test[["record_index", "Re"]]

    model_2 = LinearRegression()
    model_2.fit(X2_train, y_train)
    pred_2 = model_2.predict(X2_test)

    # Model 3: Polynomial Cycle + Re
    model_3 = Pipeline([
        ("poly", PolynomialFeatures(
            degree=2,
            include_bias=False
        )),
        ("regression", LinearRegression())
    ])

    model_3.fit(X2_train, y_train)
    pred_3 = model_3.predict(X2_test)

    predictions = {
        "Cycle only": pred_1,
        "Cycle + Re": pred_2,
        "Polynomial": pred_3
    }

    for model_name, predictions_test in predictions.items():

        comparison_results.append({
            "battery": battery_id,
            "model": model_name,
            "MAE": mean_absolute_error(
                y_test, predictions_test
            ),
            "RMSE": np.sqrt(
                mean_squared_error(y_test, predictions_test)
            ),
            "R2": r2_score(
                y_test, predictions_test
            )
        })

comparison_df = pd.DataFrame(comparison_results)

display(
    comparison_df.pivot(
        index="battery",
        columns="model",
        values="MAE"
    ).round(4)
)

In [ ]:
# Number of test observations per battery
test_weights = {
    "B0005": 40,
    "B0006": 40,
    "B0007": 40,
    "B0018": 7
}

# Add observation counts
comparison_df["test_observations"] = (
    comparison_df["battery"].map(test_weights)
)

# Calculate overall model performance
overall_results = (
    comparison_df
    .groupby("model")
    .apply(
        lambda group: pd.Series({
            "weighted_MAE": np.average(
                group["MAE"],
                weights=group["test_observations"]
            ),
            "mean_battery_MAE": group["MAE"].mean(),
            "weighted_RMSE": np.sqrt(
                np.average(
                    group["RMSE"] ** 2,
                    weights=group["test_observations"]
                )
            ),
            "total_test_observations":
                group["test_observations"].sum()
        }),
        include_groups=False
    )
    .sort_values("weighted_MAE")
)

display(overall_results.round(4))

In [ ]:
# Inspect the Cycle + Re model for each battery

selected_results = []

for battery_id in battery_ids:

    train_data = model_dataset_B[
        (model_dataset_B["battery_id"] == battery_id)
    ].copy()

    X = train_data[["record_index", "Re"]]
    y = train_data["SoH_percent"]

    model = LinearRegression()
    model.fit(X, y)

    selected_results.append({
        "battery": battery_id,
        "cycle_coefficient": model.coef_[0],
        "Re_coefficient": model.coef_[1],
        "intercept": model.intercept_
    })

coefficients_df = pd.DataFrame(selected_results)

display(coefficients_df.round(4))

In [ ]:
# Calculate coefficients using only the temporal training data

training_coefficients = []

for battery_id in battery_ids:

    battery_data = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    split_index = int(len(battery_data) * 0.7)
    train_data = battery_data.iloc[:split_index]

    X_train = train_data[["record_index", "Re"]]
    y_train = train_data["SoH_percent"]

    model = LinearRegression()
    model.fit(X_train, y_train)

    training_coefficients.append({
        "battery": battery_id,
        "training_observations": len(train_data),
        "cycle_coefficient": model.coef_[0],
        "Re_coefficient": model.coef_[1],
        "intercept": model.intercept_
    })

training_coefficients_df = pd.DataFrame(training_coefficients)

display(training_coefficients_df.round(4))

In [ ]:
# Check correlation between cycle number and Re

cycle_re_correlations = []

for battery_id in battery_ids:

    battery_data = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    split_index = int(len(battery_data) * 0.7)
    train_data = battery_data.iloc[:split_index]

    correlation = train_data[
        ["record_index", "Re"]
    ].corr().iloc[0, 1]

    cycle_re_correlations.append({
        "battery": battery_id,
        "training_observations": len(train_data),
        "cycle_Re_correlation": correlation
    })

cycle_re_corr_df = pd.DataFrame(cycle_re_correlations)

display(cycle_re_corr_df.round(4))

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Calculate VIF for each battery's training data

vif_results = []

for battery_id in battery_ids:

    battery_data = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    split_index = int(len(battery_data) * 0.7)
    train_data = battery_data.iloc[:split_index]

    X = train_data[["record_index", "Re"]].copy()

    # Standardise variables for numerical stability
    X = (X - X.mean()) / X.std()

    vif_cycle = variance_inflation_factor(X.values, 0)
    vif_re = variance_inflation_factor(X.values, 1)

    vif_results.append({
        "battery": battery_id,
        "VIF_cycle": vif_cycle,
        "VIF_Re": vif_re
    })

vif_df = pd.DataFrame(vif_results)

display(vif_df.round(2))

In [ ]:
# Test whether resistance deviations add information
# beyond the general cycle-related resistance trend.

from sklearn.linear_model import LinearRegression

residual_results = []

for battery_id in battery_ids:

    battery_data = (
        model_dataset_B[
            model_dataset_B["battery_id"] == battery_id
        ]
        .sort_values("record_index")
        .copy()
    )

    split_index = int(len(battery_data) * 0.7)

    train_data = battery_data.iloc[:split_index].copy()
    test_data = battery_data.iloc[split_index:].copy()

    # Step 1: Estimate expected Re from cycle number
    resistance_trend = LinearRegression()

    resistance_trend.fit(
        train_data[["record_index"]],
        train_data["Re"]
    )

    # Step 2: Calculate resistance residuals
    train_data["Re_residual"] = (
        train_data["Re"] -
        resistance_trend.predict(train_data[["record_index"]])
    )

    test_data["Re_residual"] = (
        test_data["Re"] -
        resistance_trend.predict(test_data[["record_index"]])
    )

    # Step 3: Fit SoH models
    model_cycle = LinearRegression()
    model_residual = LinearRegression()

    model_cycle.fit(
        train_data[["record_index"]],
        train_data["SoH_percent"]
    )

    model_residual.fit(
        train_data[["record_index", "Re_residual"]],
        train_data["SoH_percent"]
    )

    # Step 4: Evaluate on the later observations
    pred_cycle = model_cycle.predict(
        test_data[["record_index"]]
    )

    pred_residual = model_residual.predict(
        test_data[["record_index", "Re_residual"]]
    )

    residual_results.append({
        "battery": battery_id,
        "test_observations": len(test_data),
        "Cycle_only_MAE": mean_absolute_error(
            test_data["SoH_percent"], pred_cycle
        ),
        "Cycle_plus_residual_Re_MAE": mean_absolute_error(
            test_data["SoH_percent"], pred_residual
        )
    })

residual_comparison_df = pd.DataFrame(residual_results)

display(residual_comparison_df.round(4))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

project_folder = "/content/drive/MyDrive/Battery_Digital_Twin"

results_folder = os.path.join(project_folder, "results")

os.makedirs(results_folder, exist_ok=True)

print("Project folder:", project_folder)
print("Results folder:", results_folder)
print("Folder created successfully!")

In [ ]:
import os

# Results to export
results_to_save = {
    "temporal_model_comparison": comparison_df,
    "overall_model_performance": overall_results,
    "full_data_coefficients": coefficients_df,
    "training_coefficients": training_coefficients_df,
    "cycle_Re_correlations": cycle_re_corr_df,
    "multicollinearity_VIF": vif_df,
    "residual_Re_comparison": residual_comparison_df
}

# Save each result as a CSV file
for filename, dataframe in results_to_save.items():
    filepath = os.path.join(
        results_folder,
        f"{filename}.csv"
    )
    dataframe.to_csv(filepath, index=True)
    print("Saved:", filename + ".csv")

print("\nAll modelling results exported successfully!")

In [ ]:
report = """
# Battery Digital Twin – Modelling Results

## 1. Dataset
- NASA Battery Aging Dataset
- Batteries: B0005, B0006, B0007, B0018
- Total discharge observations: 636
- Matched impedance observations: 419
- Health target: State of Health (SoH), calculated from
  discharge capacity relative to initial measured capacity.

## 2. Modelling objective
Investigate whether electrical and thermal indicators,
particularly impedance resistance (Re), provide useful
information for battery SoH estimation.

## 3. Temporal validation
A chronological 70/30 train-test split was used for
each battery.

| Model | Weighted MAE | Weighted RMSE |
|---|---:|---:|
| Cycle + Re | 3.0894 | 3.3447 |
| Cycle only | 3.6955 | 4.0590 |
| Polynomial | 4.2225 | 6.0997 |

Cycle + Re achieved a 16.4% reduction in weighted MAE
compared with Cycle only.

## 4. Multicollinearity
Training-set correlations between cycle number and Re:
- B0005: 0.9629
- B0006: 0.9851
- B0007: 0.9391
- B0018: 0.2091

VIF values:
- B0005: 13.74
- B0006: 33.83
- B0007: 8.47
- B0018: 1.05

These results indicate substantial multicollinearity
for the first three batteries.

## 5. Residual resistance experiment
Resistance was modelled as a function of cycle number.
The residual resistance was then used as an additional
predictor of SoH.

| Battery | Cycle MAE | Cycle + residual Re MAE |
|---|---:|---:|
| B0005 | 2.9301 | 2.9942 |
| B0006 | 5.1297 | 3.1757 |
| B0007 | 3.1091 | 3.1378 |
| B0018 | 3.2241 | 2.8648 |

The improvement was battery-dependent and was particularly
pronounced for B0006.

## 6. Main conclusions
- Cycle number provides a useful indicator of degradation.
- Re contains potentially useful additional information.
- The relationship between Re and SoH is not consistent
  across all four batteries.
- Multicollinearity complicates coefficient interpretation.
- Polynomial regression showed weaker temporal
  generalisation than the linear alternatives.
- The extended resistance-based model performed better
  overall, but further validation is necessary.

## 7. Limitations
- Only four battery cells were investigated.
- B0018 had only seven temporal test observations.
- Different discharge cut-off voltages affect
  cross-battery comparability.
- Temporal extrapolation is challenging because later
  degradation states differ from training conditions.
- The current model is empirical, not yet physics-informed.
- The current estimator uses cycle-level measurements
  after discharge, not pre-cycle forecasting.

## 8. Next development stage
Develop a Digital Twin architecture with:
- Battery state representation
- Cycle-by-cycle state updates
- Resistance-based health indicators
- SoH estimation component
- Monitoring and visualisation dashboard
"""

report_path = os.path.join(
    results_folder,
    "modelling_summary.md"
)

with open(report_path, "w") as file:
    file.write(report)

print("Report saved:", report_path)

In [ ]:
notebooks_folder = os.path.join(
    project_folder,
    "notebooks"
)

os.makedirs(notebooks_folder, exist_ok=True)

print("Notebook folder created:")
print(notebooks_folder)

In [ ]:
# Export the combined battery dataset

combined_data_path = os.path.join(
    results_folder,
    "combined_battery_dataset.csv"
)

combined_df.to_csv(combined_data_path, index=False)

print("Dataset exported successfully!")
print("File:", combined_data_path)
print("Shape:", combined_df.shape)

In [ ]:
import os
import glob

print("Current working directory:")
print(os.getcwd())

print("\nMAT files in current directory:")
for file in glob.glob("*.mat"):
    print(os.path.abspath(file))

print("\nZIP files in current directory:")
for file in glob.glob("*.zip"):
    print(os.path.abspath(file))

In [ ]:
import os
import shutil
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create a folder for the original NASA data
project_folder = "/content/drive/MyDrive/Battery_Digital_Twin"
raw_data_folder = os.path.join(project_folder, "raw_data")

os.makedirs(raw_data_folder, exist_ok=True)

# Copy all four original battery files
battery_files = [
    "B0005.mat",
    "B0006.mat",
    "B0007.mat",
    "B0018.mat"
]

for file in battery_files:
    source = os.path.join("/content", file)
    destination = os.path.join(raw_data_folder, file)

    shutil.copy2(source, destination)

    print(f"Copied: {file}")

print("\nAll original files have been saved successfully!")
print("Destination:", raw_data_folder)